In [0]:
%python """Read CSV with spark"""
customers_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("/Volumes/workspace/bronze/raw_data_volume/ecommerce_dataset/olist_customers_dataset.csv")
)
(
    customers_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("bronze.customers")
)


1. Compare Data count between src and tgt


In [0]:
#  Validate After Full-Refresh , Method -1
print("Landing Layer data Count :", customers_df.count())
print("Bronze Layer data Count :", spark.table("bronze.customers").count())

2. Test Distinct PK on src and tgt


In [0]:
source_pk = customers_df.select("customer_id").distinct().count()

target_pk = (
    spark.table("bronze.customers")
         .select("customer_id")
         .distinct()
         .count()
)

print(source_pk)
print(target_pk)

3. Data Comparison using Except

In [0]:
print(customers_df.exceptAll(
    spark.table("bronze.customers")
).count())

spark.table("bronze.customers").exceptAll(
    customers_df
).count()

4. Full Refresh Replacement Test

In [0]:
before_count = spark.table("bronze.customers").count()
print(before_count)

In [0]:
# Create a dataset 
from pyspark.sql import Row
from pyspark.sql import functions as F
dummy = spark.createDataFrame([
    Row(
        customer_id="TEST123",
        customer_unique_id="TEST123",
        customer_zip_code_prefix=111111,
        customer_city="PUNE",
        customer_state="MH"
    )
])
dummy = dummy.withColumn(
    "customer_zip_code_prefix",
    F.col("customer_zip_code_prefix").cast("int")
)
test_df = customers_df.unionByName(dummy)
print(test_df.count())
# customers_df.printSchema()

In [0]:
(
    test_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze.customers")
)


In [0]:
spark.table("bronze.customers") \
     .filter("customer_id='TEST123'") \
     .show()

In [0]:
(
    customers_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze.customers")
)

spark.table("bronze.customers") \
     .filter("customer_id='TEST123'") \
     .show()